# Run in colab

In [1]:
!pip install instructor

In [2]:
!cd /content
!rm -rf macro_financial_forecasting

In [3]:
!git clone https://github.com/chuanbinp/macro_financial_forecasting.git

Cloning into 'macro_financial_forecasting'...
remote: Enumerating objects: 1586, done.
remote: Counting objects: 100% (575/575), done.
remote: Compressing objects: 100% (184/184), done.
remote: Total 1586 (delta 427), reused 397 (delta 391), pack-reused 1011 (from 1)
Receiving objects: 100% (1586/1586), 19.07 MiB | 40.11 MiB/s, done.
Resolving deltas: 100% (958/958), done.


In [4]:
%cd macro_financial_forecasting/applications/macro_financial_forecasting/src

/content/macro_financial_forecasting/applications/macro_financial_forecasting/src


In [5]:
from config import Config
from train_data_loader import TrainDataLoader
# from agentics_transducer import AgenticTransducer
from data_model.bloomberg_news_entry import BloombergNewsEntry
from google.colab import userdata
import os

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
config = Config("../config.env")

train_data_loader = TrainDataLoader(config)

In [6]:
print("Starting loading pipeline ...")
print(f"Config: {config}")

train_ds = train_data_loader.load()
print("Loading pipeline completed.")

Starting loading pipeline ...
Config: Config(
  gemini_api_key: !secret!
  openai_api_key: !secret!
  llm_model: openai/gpt-5-nano-2025-08-07
  industries: ['Information Technology', 'Health Care', 'Financials', 'Consumer Discretionary', 'Communication Services', 'Industrials', 'Consumer Staples', 'Energy', 'Utilities', 'Real Estate', 'Materials', 'General Market', 'None']
  dataset_name: danidanou/Bloomberg_Financial_News
  dataset_dir: ../data/
  rss_feeds: ['https://feeds.bloomberg.com/news/news.rss', 'https://feeds.bloomberg.com/markets/news.rss', 'https://feeds.bloomberg.com/business/news.rss', 'https://feeds.bloomberg.com/technology/news.rss', 'https://feeds.bloomberg.com/politics/news.rss', 'https://feeds.bloomberg.com/wealth/news.rss', 'https://feeds.bloomberg.com/economics/news.rss', 'https://feeds.bloomberg.com/green/news.rss', 'https://feeds.bloomberg.com/pursuits/news.rss', 'https://feeds.bloomberg.com/opinion/news.rss', 'https://feeds.bloomberg.com/finance/news.rss', 'http

bloomberg_financial_data.parquet.gzip:   0%|          | 0.00/482M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/446762 [00:00<?, ? examples/s]


--- Download Successful! ---


Map:   0%|          | 0/446762 [00:00<?, ? examples/s]

Training dataset processed.

--- Starting validation of 446762 entries ---


Validating entries: 100%|██████████| 446762/446762 [00:24<00:00, 18326.80it/s]


--- Validation Complete! ---
Training dataset validated.
Saving processed dataset to local cache at '../data/danidanou_Bloomberg_Financial_News_train'...
Total number of rows: 446762
Loading pipeline completed.


In [7]:
train_ds[0]

BloombergNewsEntry(Headline='Ivory Coast Keeps Cocoa Export Tax Below 22%, Document Shows', Date='2011-10-06', Link='http://www.bloomberg.com/news/2011-10-06/ivory-coast-keeps-cocoa-export-tax-below-22-document-shows.html', Article='Export taxes on cocoa beans from Ivory Coast , the world’s biggest producer of the chocolate ingredient, won’t exceed 22 percent of the international price this season, meeting a commitment to the International Monetary Fund , according to a finance ministry document. In the 2008-9 season taxes averaged 25.3 percent of international prices, the IMF said in a document posted on its website in November last year. While the country met the commitment in the season just ended, it had a change in government earlier this year. The rate meets a demand by the International Monetary Fund and the World Bank to reform the Ivorian cocoa and coffee industries in order to comply with the terms of its Heavily Indebted Poor Countries’ debt-relief program. Last year, the fi

In [8]:
from processor import NewsProcessor
import nest_asyncio
nest_asyncio.apply()

processor = NewsProcessor(config)
results = await processor.transduce_news_entries_async(train_ds[:50], save_path_prefix="processed_news") #Sample 50 news

Processing news: 100%|██████████| 50/50 [00:38<00:00,  1.31entry/s]


In [9]:
processed_df = processor.group_by_date_and_industry(results, save_path="grouped_data")

In [10]:
processed_df

,Industry,Date,News
0,Communication Services,2011-10-06,[{'Headline': 'New York Times Climbs After Mex...
1,Consumer Discretionary,2011-10-06,[{'Headline': 'Breguet Sales to Exceed 500 Mil...
2,Consumer Staples,2011-10-06,[{'Headline': 'PepsiCo May Purchase Russian Dr...
3,Energy,2011-10-06,[{'Headline': 'Lukoil May Invest $300 Mln in R...
4,Financials,2011-10-06,[{'Headline': 'France Working on Contingency B...
5,Financials,2011-10-07,[{'Headline': 'Hana Financial Pushes for Korea...
6,General Market,2011-10-06,"[{'Headline': 'Hungary Focusing on Debt, Defic..."
7,General Market,2011-10-07,[{'Headline': 'Vietnam Raises a Key Rate to Su...
8,Industrials,2011-10-06,[{'Headline': 'Airbus German Workers Plan Work...
9,Information Technology,2011-10-06,[{'Headline': 'RadVision Advances Most in a We...


In [11]:
processed_df = await processor.process_dataframe(processed_df, save_path="sentiment_data")

Sentiment & Explanation: 100%|██████████| 13/13 [00:20<00:00,  1.55s/it]


In [12]:
processed_df

,Industry,Date,News,Summary,SentimentScore,SentimentExplanation
0,Communication Services,2011-10-06,[{'Headline': 'New York Times Climbs After Mex...,- Swatch Group's Breguet brand is expected to ...,0.25,The Consumer Discretionary sector shows a posi...
1,Consumer Discretionary,2011-10-06,[{'Headline': 'Breguet Sales to Exceed 500 Mil...,- Hungary to tighten 2012 budget to reduce deb...,0.25,Moderately positive stance for Consumer Staple...
2,Consumer Staples,2011-10-06,[{'Headline': 'PepsiCo May Purchase Russian Dr...,- The State Bank of Vietnam raised the refinan...,-0.25,Vietnam's rate hike and dong depreciation occu...
3,Energy,2011-10-06,[{'Headline': 'Lukoil May Invest $300 Mln in R...,- Walford on Shrewsbury Road is listed for 15 ...,0.25,The New York Times Co. stock rose 13% in a sin...
4,Financials,2011-10-06,[{'Headline': 'France Working on Contingency B...,- Slim's family vehicle Inmobiliaria Carso boo...,0.20,Mixed but leaning positive for Real Estate: U....
5,General Market,2011-10-06,"[{'Headline': 'Hungary Focusing on Debt, Defic...",- The Uganda Coffee Development Authority quot...,-0.60,Irish trophy-home markets show extreme weaknes...
6,Industrials,2011-10-06,[{'Headline': 'Airbus German Workers Plan Work...,- France contingency plan to take stakes in 2-...,0.30,The Korea financials story is modestly positiv...
7,Information Technology,2011-10-06,[{'Headline': 'RadVision Advances Most in a We...,- Airbus Germany: IG Metall calls for work sto...,0.25,The Information Technology sector shows a mild...
8,Materials,2011-10-06,[{'Headline': 'Uganda Coffee Development Autho...,Summary\n- Lone Star found guilty of stock-pri...,0.25,Overall General Market sentiment is mildly pos...
9,Real Estate,2011-10-06,[{'Headline': 'Australian Homes for Sale Jump ...,- Lukoil may invest up to $300 million in offs...,-0.15,Industrial sector sentiment is mildly negative...


In [14]:
import pandas as pd

pd.read_parquet("../data/sentiment_data")

,Industry,Date,News,Summary,SentimentScore,SentimentExplanation
0,Communication Services,2011-10-06,[{'Article': 'New York Times Co. (NYT) climbed...,- Swatch Group's Breguet brand is expected to ...,0.25,The Consumer Discretionary sector shows a posi...
1,Consumer Discretionary,2011-10-06,[{'Article': 'Swatch Group AG (UHR) ’s Breguet...,- Hungary to tighten 2012 budget to reduce deb...,0.25,Moderately positive stance for Consumer Staple...
2,Consumer Staples,2011-10-06,[{'Article': 'PepsiCo Inc. is in talks to buy ...,- The State Bank of Vietnam raised the refinan...,-0.25,Vietnam's rate hike and dong depreciation occu...
3,Energy,2011-10-06,"[{'Article': 'OAO Lukoil, Russia ’s second-big...",- Walford on Shrewsbury Road is listed for 15 ...,0.25,The New York Times Co. stock rose 13% in a sin...
4,Financials,2011-10-06,[{'Article': 'France is working on a contingen...,- Slim's family vehicle Inmobiliaria Carso boo...,0.20,Mixed but leaning positive for Real Estate: U....
5,General Market,2011-10-06,[{'Article': 'Hungary’s government is embarkin...,- The Uganda Coffee Development Authority quot...,-0.60,Irish trophy-home markets show extreme weaknes...
6,Industrials,2011-10-06,[{'Article': 'Airbus SAS labor unions in Germa...,- France contingency plan to take stakes in 2-...,0.30,The Korea financials story is modestly positiv...
7,Information Technology,2011-10-06,[{'Article': 'RadVision Ltd. (RVSN) jumped the...,- Airbus Germany: IG Metall calls for work sto...,0.25,The Information Technology sector shows a mild...
8,Materials,2011-10-06,[{'Article': 'The Uganda Coffee Development Au...,Summary\n- Lone Star found guilty of stock-pri...,0.25,Overall General Market sentiment is mildly pos...
9,Real Estate,2011-10-06,[{'Article': 'Australian properties listed for...,- Lukoil may invest up to $300 million in offs...,-0.15,Industrial sector sentiment is mildly negative...
